# 00 - OCI Generative AI Setup and Authentication

This notebook is the shared starting point for the three integration guides:

1. Oracle Database 26ai + OCI Generative AI using the OCI SDK
2. LangChain + OCI Generative AI using `langchain-oci`
3. n8n + OCI Generative AI using an OpenAI-compatible gateway

The goal is not to build an application yet. The goal is to make sure everyone understands the basic OCI concepts, has local authentication configured, and can make one small OCI Generative AI request before moving to the guide-specific notebooks.

Official references used for this setup:

- OCI Generative AI documentation: https://docs.oracle.com/en-us/iaas/Content/generative-ai/home.htm
- OCI API key authentication for Generative AI: https://docs.oracle.com/en-us/iaas/Content/generative-ai/setup-oci-api-auth.htm
- OCI Python SDK config file authentication: https://docs.oracle.com/en-us/iaas/Content/API/Concepts/sdkconfig.htm
- OCI Python SDK Generative AI Inference examples: https://docs.oracle.com/en-us/iaas/tools/python-sdk-examples/latest/generativeaiinference/


Copyright (c) 2026 Oracle and/or its affiliates. Licensed under the Universal Permissive License (UPL), Version 1.0.


## What You Will Have At The End

By the end of this notebook, you should have:

- A local OCI config file that Python can read.
- A `.env` file with the required configuration values.
- A working OCI Generative AI Inference client.
- A successful small test request to a chat model.

This is the same foundation used by the written guides. The later notebooks reuse these settings instead of explaining them again from scratch.


## The Few OCI Terms We Need

| Term | Plain-language meaning | Where it appears in the notebooks |
|---|---|---|
| Tenancy | Your organization's OCI cloud account | Created before running these examples |
| Region | The OCI data center region you call | `GENAI_ENDPOINT`, OCI config |
| Compartment | A folder-like boundary for OCI resources and permissions | `OCI_COMPARTMENT_ID` |
| OCID | Oracle Cloud Identifier; a unique ID for a resource | compartment, user, tenancy IDs |
| IAM policy | Rule that grants access to OCI services | required for `generative-ai-family` |
| API key auth | Local private key + config profile used by SDKs | `~/.oci/config` |
| Model ID | Name or OCID of the model to call | `CHAT_MODEL_ID`, `EMBED_MODEL_ID` |
| Endpoint | URL for the Generative AI inference service in your region | `GENAI_ENDPOINT` |


## OCI API Authentication Setup

This notebook uses OCI API key authentication, the same pattern used by the OCI SDK and CLI when they run from your laptop or a notebook environment.

The setup flow is:

1. In the OCI Console, open your user settings and add an API key under **Tokens and keys**.
2. Generate or upload an API signing key pair. Keep the private key local and store it somewhere secure, commonly under `~/.oci/`.
3. Copy the configuration snippet that the Console shows after the key is added into a local config file, usually `~/.oci/config`.
4. Make sure the profile contains `user`, `fingerprint`, `tenancy`, `region`, and `key_file`. The `key_file` value must point to the private key on your machine.
5. Set `OCI_CONFIG_FILE` and `OCI_PROFILE` in `.env` so the notebook loads the same profile you created.

The config file proves who is making the request. Authorization is still controlled by IAM policies, compartment access, model availability, and the region you call. The later cells validate both sides: first the local config profile, then the Generative AI endpoint and model request.


## Install Python Packages

The setup notebook only needs the OCI Python SDK and `python-dotenv`. Later notebooks add their own topic-specific packages.


In [ ]:
# Dependencies are managed by uv. Run `uv sync` from the files directory before opening this notebook.

## Create Your `.env` File

Copy the repository's `.env.example` file to `.env`, then replace the placeholder values.

The minimum values used by this notebook are:

```text
OCI_PROFILE=DEFAULT
OCI_CONFIG_FILE=~/.oci/config
OCI_COMPARTMENT_ID=replace_with_compartment_ocid
GENAI_ENDPOINT=https://inference.generativeai.<region>.oci.oraclecloud.com
CHAT_MODEL_ID=<chat-model-available-in-your-region>
CHAT_REQUEST_FORMAT=COHERE
CHAT_MAX_TOKENS=1024
CHAT_MAX_COMPLETION_TOKENS=8192
CHAT_REASONING_EFFORT=LOW
```


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
import sys

# Make the repo-level helper package importable from this notebook.
for directory in [Path.cwd(), *Path.cwd().parents]:
    if (directory / "workshop_helpers").exists():
        sys.path.insert(0, str(directory))
        break
else:
    raise FileNotFoundError("Could not find the workshop_helpers folder.")

from workshop_helpers.oci_genai_helpers import load_workshop_env

# Load tenancy-specific settings such as compartment OCID, endpoint, and model IDs.
env_path = load_workshop_env()
display(Markdown(f"""
**Environment Loaded**

`{env_path}`
"""))


## What The OCI Config File Looks Like

The earlier API-auth section described the Console-generated config snippet. The OCI SDK reads that snippet from a local config file, usually at `~/.oci/config`. A profile looks like this:

```ini
[DEFAULT]
user=replace_with_user_ocid
fingerprint=replace_with_key_fingerprint
tenancy=replace_with_tenancy_ocid
region=<region>
key_file=/Users/you/.oci/oci_api_key.pem
```

Only the public key is registered with OCI. The SDK uses the private key locally to sign API requests, so do not paste the private key contents into `.env` or the notebook.


In [ ]:
import os

from workshop_helpers.oci_genai_helpers import load_oci_config, mask

# Load the OCI API-key profile from ~/.oci/config, then validate the profile.
oci_config_file = os.path.expanduser(os.getenv("OCI_CONFIG_FILE", "~/.oci/config"))
oci_profile = os.getenv("OCI_PROFILE", "DEFAULT")
config = load_oci_config()

# Display only safe setup details. OCIDs are masked because this notebook may be shared.
display(Markdown(f"""
**OCI Config Loaded**

| Setting | Value |
|---|---|
| Profile | `{oci_profile}` |
| Config file | `{oci_config_file}` |
| Region | `{config.get('region')}` |
| Tenancy | `{mask(config.get('tenancy'))}` |
| User | `{mask(config.get('user'))}` |
| Key file | `{config.get('key_file')}` |
"""))


## Check The Workshop Variables

This cell checks the values that the three guide notebooks will reuse. It does not call OCI yet.


In [ ]:
import os

# These values are required before any OCI Generative AI request can work.
required_vars = [
    "OCI_COMPARTMENT_ID",
    "GENAI_ENDPOINT",
    "CHAT_MODEL_ID",
]

# These values are reused by later notebooks, but some have safe defaults.
optional_vars = [
    "EMBED_MODEL_ID",
    "CHAT_REQUEST_FORMAT",
    "CHAT_MAX_TOKENS",
    "CHAT_MAX_COMPLETION_TOKENS",
    "CHAT_REASONING_EFFORT",
]

rows = []
missing = []
for name in required_vars:
    value = os.getenv(name)
    status = "OK" if value and "replace" not in value.lower() and "<region>" not in value else "MISSING"
    rows.append((name, status, mask(value, 32)))
    if status == "MISSING":
        missing.append(name)

for name in optional_vars:
    value = os.getenv(name)
    rows.append((name, "SET" if value else "OPTIONAL", value or ""))

# Show the environment check as a table so the configuration is easy to inspect.
table_rows = "\n".join(f"| `{name}` | {status} | `{value}` |" for name, status, value in rows)
display(Markdown(f"""
**Configuration Variable Check**

| Variable | Status | Value |
|---|---:|---|
{table_rows}
"""))

if missing:
    raise ValueError(f"Update these values in .env before continuing: {', '.join(missing)}")


## Create The OCI Generative AI Client

This mirrors the written guide's first code block: load OCI config, then create `GenerativeAiInferenceClient` with a region-specific service endpoint.


In [ ]:
from workshop_helpers.oci_genai_helpers import create_genai_client

# Create the OCI Generative AI client using the same config-file auth pattern as the written guide.
genai_client = create_genai_client()
display(Markdown("""
**Client Created**

OCI Generative AI Inference client is ready.
"""))


## Make One Small Chat Request



In [ ]:
from workshop_helpers.oci_genai_helpers import chat_once

# Send one small prompt to confirm auth, endpoint, compartment, and model ID are all correct.
prompt = "In one sentence, explain what OCI Generative AI is."
answer = chat_once(genai_client, prompt)

display(Markdown(f"""
**Prompt**

> {prompt}

**Model Response**

{answer}
"""))


## Quick Troubleshooting

| Symptom | Likely cause | What to check |
|---|---|---|
| `ConfigFileNotFound` | SDK cannot find OCI config | `OCI_CONFIG_FILE`, `~/.oci/config` |
| `InvalidPrivateKey` | Private key path or format issue | `key_file` in OCI config |
| `NotAuthorizedOrNotFound` | IAM policy, compartment, or model access issue | `generative-ai-family` policy, compartment OCID, model region |
| Endpoint connection error | Wrong region endpoint | `GENAI_ENDPOINT` matches config region |
| Model or request error | Model does not match request format | use Cohere model with `COHERE`, or set `CHAT_REQUEST_FORMAT=GENERIC` |
| JSON response but no text | Completion budget used before visible text was produced | increase `CHAT_MAX_COMPLETION_TOKENS`; for Gemini try `8192` or `16384` and keep `CHAT_REASONING_EFFORT=LOW` |
| Placeholder value error | `.env` was copied but not edited | replace every `replace_me`, `<region>`, and placeholder model ID |

A useful independent auth test is running this in a terminal after installing OCI CLI:

```bash
oci os ns get --profile <your-profile>
```


## Ready For The Guide Notebooks

Once the final chat cell works, the shared OCI setup is ready.

Next notebooks will reuse this same configuration:

- Guide 1 uses the same `create_genai_client()` pattern, then adds embeddings and Oracle Database 26ai vector search.
- Guide 2 uses the same profile, endpoint, compartment, and model values through `langchain-oci`.
- Guide 3 uses the same OCI authentication behind an OpenAI-compatible gateway for n8n.
